# Product dimentions table generation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
products = spark.table("ecommerce_lakehouse.silver.products")

In [0]:
products.limit(5).display()

### Check uniquness

In [0]:
products.createTempView("products_view")

In [0]:
%sql
select count(*) == count(distinct product_id) from products_view

### Add serogate key - SK

In [0]:
products = products.withColumn("product_sk", F.row_number().over(Window.orderBy("product_id")))
products.limit(5).display()

### Remove lineage columns _load_timestamp and _source_file


In [0]:
products = products.drop("_load_timestamp").drop("_source_file")

In [0]:
products.limit(5).display()

### Write to gold 

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.gold")

In [0]:
products.write.format("delta").mode("overwrite").option("inferSchema", True).saveAsTable("ecommerce_lakehouse.gold.dim_products")

In [0]:
spark.table("ecommerce_lakehouse.gold.dim_products").limit(5).display()